## 1. Document basics
Creates one `Document` by hand to show LangChain's format: `page_content` (the text) and `metadata` (info about it, like source). Every loader returns objects like this.

In [ ]:
# Import the Document class from LangChain Core
from langchain_core.documents import Document

# Manually create a document structure to understand its format (page_content & metadata)
doc = Document(
    page_content="This is the main text content I'm using to create RAG",
    metadata={
        "source": "example.txt",
        "total_pages": 1,
        "author": "harshai",
        "date_created": "2026-07-29"
    }
)

print(doc)

page_content='This is the main text content I'm using to create RAG' metadata={'source': 'example.txt', 'total_pages': 1, 'author': 'krishnaik', 'date_created': '2024-01-01'}


## 2. List the text files
Looks inside `data/txt_files` and prints the name of every `.txt` file, just to confirm the data is there.

In [ ]:
# Sample text files already live in data/txt_files - list them
from pathlib import Path

for file_path in Path("data/txt_files").glob("*.txt"):
    print(file_path.name)

## 3. Load one file
`TextLoader` reads a single file (`sample.txt`) into a list of `Document` objects.

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/txt_files/sample.txt", encoding="utf-8")
document = loader.load()
print(document)



[Document(metadata={'source': 'data/txt_files/sample.txt'}, page_content='\n    Python is a high-level, general-purpose programming language that emphasizes\n code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.\nGuido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5,[39] capabilities and keywords for typing were added to the language, allowing optional static typing.[40] As of 2026, the Python Software Foundation supports Python 3.11, 3.12, 3.13, 3.14, and 3.15, following the project\'s annual release cycle and five-year support policy. Python 3.15.0 the latest s

## 4. Load the whole folder
`DirectoryLoader` reads every `.txt` file in the folder with `TextLoader` and prints how many documents were loaded.

In [8]:
# Import DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader

# Configure DirectoryLoader to batch-read all text files from a folder
dir_loader = DirectoryLoader(
    path="data/txt_files/",
    glob="**/*.txt",          # Match all .txt files recursively
    loader_cls=TextLoader,    # Use TextLoader class for parsing
    show_progress=False
)

# Load documents
text_documents = dir_loader.load()
print(f"Loaded {len(text_documents)} text documents.")

Loaded 2 text documents.


## 4b. Load all file types
Reads `.txt`, `.pdf`, `.docx`, `.html`, `.pptx` and `.xlsx` files from the `data` folder. Each file becomes one or more `Document` objects, saved in `docs`.

In [28]:
from pathlib import Path
from bs4 import BeautifulSoup                      # reads HTML
from docx import Document as DocxDocument          # reads Word (renamed to avoid a name clash)
from pptx import Presentation                      # reads PowerPoint
from openpyxl import load_workbook                 # reads Excel
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader


# ---------- One small loader per file type ----------
def load_txt(path):
    # plain text -> 1 Document
    return TextLoader(str(path), encoding="utf-8").load()

def load_pdf(path):
    # PDF -> 1 Document per page (the page number is kept in metadata)
    return PyPDFLoader(str(path)).load()

def load_docx(path):
    # Word -> 1 Document with all non-empty paragraphs
    word_file = DocxDocument(str(path))
    text = "\n".join(p.text for p in word_file.paragraphs if p.text.strip())
    return [Document(page_content=text, metadata={"source": str(path)})]

def load_html(path):
    # HTML -> 1 Document with the visible text only (scripts and styles removed)
    html = path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    return [Document(page_content=text, metadata={"source": str(path)})]

def load_pptx(path):
    # PowerPoint -> 1 Document per slide that has text
    slides = Presentation(str(path)).slides
    docs = []
    for number, slide in enumerate(slides, start=1):
        parts = [
            shape.text_frame.text
            for shape in slide.shapes
            if shape.has_text_frame and shape.text_frame.text.strip()
        ]
        if parts:
            docs.append(Document(
                page_content="\n".join(parts),
                metadata={"source": str(path), "slide": number},
            ))
    return docs

def load_xlsx(path):
    # Excel -> 1 Document per sheet; each row becomes one line: "cell | cell | cell"
    workbook = load_workbook(str(path), data_only=True, read_only=True)
    docs = []
    for sheet in workbook.worksheets:
        lines = []
        for row in sheet.iter_rows(values_only=True):
            cells = [str(cell) for cell in row if cell is not None]   # skip empty cells
            if cells:
                lines.append(" | ".join(cells))
        if lines:
            docs.append(Document(
                page_content="\n".join(lines),
                metadata={"source": str(path), "sheet": sheet.title},
            ))
    workbook.close()
    return docs


# ---------- Match each file extension to its loader ----------
# ---------- Match each file extension to its loader ----------
LOADERS = {
    ".txt": load_txt,
    ".md": load_txt,       
    ".pdf": load_pdf,
    ".docx": load_docx,
    ".html": load_html,
    ".htm": load_html,
    ".pptx": load_pptx,
    ".xlsx": load_xlsx,
}


# ---------- Go through the folder and load every supported file ----------
def load_documents(folder="data"):
    all_docs = []
    for path in sorted(Path(folder).rglob("*")):           # rglob = look inside subfolders too
        loader = LOADERS.get(path.suffix.lower())
        if loader is None:                                 # skip unsupported files
            continue
        try:
            loaded = loader(path)
            for d in loaded:
                d.metadata["file_type"] = path.suffix.lower().lstrip(".")
            all_docs.extend(loaded)
            print(f"Loaded {len(loaded):>3} document(s) from {path}")
        except Exception as e:                             # one bad file should not stop the rest
            print(f"Skipped {path}: {e}")
    return all_docs


# ---------- Run it ----------
docs = load_documents("data")
print(f"\nTotal: {len(docs)} documents")


Loaded   1 document(s) from data\Book2.xlsx
Loaded   1 document(s) from data\CLAUDE_1.md
Loaded   1 document(s) from data\txt_files\ml.txt
Loaded   1 document(s) from data\txt_files\sample.txt

Total: 4 documents


## 5. Split into chunks
Takes `docs` from the loading cell above and cuts every document into pieces of about 800 characters, with 100 characters of overlap so sentences at the edges aren't lost. The result is `chunks`, which the embedding step uses.

In [29]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size    = max characters per chunk
# chunk_overlap = characters shared between neighbours, so no sentence is lost at the cut
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

# "docs" comes from Cell 1
chunks = splitter.split_documents(docs)

print(f"{len(docs)} documents -> {len(chunks)} chunks")

# Quick look at the first chunk (text + where it came from)
print("\nFirst chunk:")
print(chunks[0].page_content[:200])
print(chunks[0].metadata)


4 documents -> 23 chunks

First chunk:
yayyyy
{'source': 'data\\Book2.xlsx', 'sheet': 'Sheet1', 'file_type': 'xlsx'}


## Embeddings
Turning text into numbers, so that texts with similar meaning get similar numbers.

### 6. EmbeddingManager
A class that loads the `all-MiniLM-L6-v2` model and has `generate_embeddings(texts)`, which turns a list of strings into a `(n, 384)` array. The last line creates `embedding_manager` for reuse.

In [7]:
import numpy as np
from typing import List
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Return a numpy array of shape (len(texts), embedding_dim)"""
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# initialize the embedding manager
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")


c:\Users\Sir\OneDrive\Desktop\rag 1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1748.68it/s]


Model loaded. Embedding dimension: 384


C:\Users\Sir\AppData\Local\Temp\ipykernel_22312\1173432364.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### 7. VectorStore
A class that wraps ChromaDB. It opens (or creates) a saved database in `data/vector_store` and has `add_documents(chunks, embeddings)` to store the text, metadata and vector of each chunk. The last line creates `vector_store`.

In [ ]:
import os
import uuid
from typing import List, Dict, Any, Tuple
import chromadb
from chromadb.config import Settings
import numpy as np

class VectorStore:
    """Handles Vector Store DB operations using ChromaDB"""
    
    def __init__(self, collection_name: str = "rag_documents", persistent_directory: str = "data/vector_store"):
        """
        Initialize the vector store and ChromaDB client
        """
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        
        # Initialize the store
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent directory if it doesn't exist
            os.makedirs(self.persistent_directory, exist_ok=True)
            
            # Initialize ChromaDB persistent client
            self.client = chromadb.PersistentClient(path=self.persistent_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "RAG pipeline vector store collection", "hnsw:space": "cosine"}
            )
            
            print(f"Vector store initialized successfully. Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
            
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their corresponding embeddings to the vector store
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings.")
            
        try:
            ids = []
            embeddings_list = []
            metadatas = []
            documents_text = []
            
            for idx, (doc, embedding) in enumerate(zip(documents, embeddings)):
                # Generate a unique ID for each record
                doc_id = f"chunk_{idx}"
                ids.append(doc_id)
                
                # Prepare metadata
                metadata = doc.metadata.copy() if hasattr(doc, 'metadata') else {}
                metadata["doc_index"] = idx
                metadata["content_length"] = len(doc.page_content) if hasattr(doc, 'page_content') else len(str(doc))
                metadatas.append(metadata)
                
                # Get text content
                content = doc.page_content if hasattr(doc, 'page_content') else str(doc)
                documents_text.append(content)
                
                # Convert numpy embedding to list for ChromaDB
                embeddings_list.append(embedding.tolist())
                
            # Add to ChromaDB collection
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

# --- Initialize and Use the Vector Store ---
vector_store = VectorStore(
    collection_name="rag_documents",
    persistent_directory="data/vector_store"
)

# Example of how to add chunks & embeddings (assuming chunks and embedding_manager are defined):
# text_list = [doc.page_content for doc in chunks]
# chunk_embeddings = embedding_manager.generate_embeddings(text_list)
# vector_store.add_documents(documents=chunks, embeddings=chunk_embeddings)

### 8. Embed and store
Takes the text out of every chunk, turns it into vectors with `embedding_manager`, and saves chunks + vectors in the vector database. Needs cells 5, 6 and 7 to have run first.

In [10]:
texts = [doc.page_content for doc in chunks]

## generate the embedding 
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
vector_store.add_documents(chunks,embeddings)


Generating embeddings for 7 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]

Generated embeddings with shape: (7, 384)
Successfully added 7 documents to vector store.
Total documents in collection: 7


## RAG retrieval pipeline
Given a question, find the chunks that are closest in meaning.

### 9. RagRetriever
A class whose `retrieve(query)` embeds the question, asks ChromaDB for the `top_k` nearest chunks, converts distance to a score and returns them as dicts. The bottom part creates `rag_retriever` and tests it with "What is machine learning?".

In [22]:

from typing import List, Dict, Any


class RagRetriever:
    def __init__(self, vector_store: Any, embedding_manager: Any):
        self.vector_store = vector_store                # the database holding your chunks
        self.embedding_manager = embedding_manager      # turns text into numbers

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        # 1. Turn the question into numbers (same model as the chunks)
        query_embedding = self.embedding_manager.generate_embeddings([query])
        query_vector = query_embedding[0].tolist()

        # 2. Ask the database for the closest chunks
        results = self.vector_store.collection.query(
            query_embeddings=[query_vector],
            n_results=min(top_k, self.vector_store.collection.count())   # never ask for more than exist
        )

        # 3. Unpack the results (ChromaDB wraps them in an extra list, so take [0])
        ids = results["ids"][0]
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        # 4. Build a clean list, keeping only chunks above the score threshold
        retrieved_docs = []
        for doc_id, doc_text, metadata, distance in zip(ids, documents, metadatas, distances):
            similarity_score = 1 - distance             # cosine distance -> similarity (higher is better)

            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "id": doc_id,
                    "content": doc_text,
                    "metadata": metadata or {},
                    "score": similarity_score
                })

        return retrieved_docs


# Build the retriever using the REAL vector_store from cell 7
rag_retriever = RagRetriever(vector_store, embedding_manager)

# Quick test
context_docs = rag_retriever.retrieve(
    "What is machine learning?",
    top_k=3,
    score_threshold=0.2
)

for doc in context_docs:
    print(f"\nScore: {doc['score']:.3f} | Source: {doc['metadata'].get('source', 'unknown')}")
    print(doc["content"])


Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.08it/s]

Generated embeddings with shape: (1, 384)

Score: 0.670 | Source: data\txt_files\ml.txt
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.[1]

Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.[3][4]

Score: 0.643 | Source: data\txt_files\ml.txt
Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance. The resulting dominance of machine learning within the field of AI from the late 2000s and early 2010s onwards was such that the terms "machine learning" and "artificial intelligence" are now often used interchangeably, e

## LLM answer pipeline
Send the retrieved text plus the question to a language model.

### 10. LLM and `rag_simple`
Loads the Groq API key from `.env` and creates the `llm`. `rag_simple` then does the full RAG: retrieve chunks, join them into a context string, build a prompt that says "answer only from this text", send it to the LLM and return the answer.

In [12]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()  # load enviroment 

### ChatGroq reads GROQ_API_KEY from the environment automatically
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

## reterive context + prompt 
def rag_simple(query,retriever,llm,top_k=3):
    ## get the closest chunks 
    results=retriever.retrieve(query,top_k=top_k)

    if results:
        context = "\n\n".join([doc["content"] for doc in results])
    else:
        context = ""


    prompt = f"""Answer using only the text below. If the answer is not there, say "I don't know".

Text:
{context}

Question:
{query}
"""

    # send prompt to the llm
    response = llm.invoke(prompt)
    return response.content

### 11. Ask a question
Runs the whole pipeline end to end for "what is ml?" and prints the answer.

In [27]:
print(rag_simple("what is global warming?", rag_retriever, llm, top_k=3))

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


Generated embeddings with shape: (1, 384)
I don't know.


12. Advance rag features

In [ ]:
def rag_advanced(query, retriever, llm, top_k=3, min_score=0.2, return_context=False):
    """
    Answer a question using retrieved chunks and report how trustworthy it is.

    Args:
        query:          the user's question
        retriever:      RagRetriever object (has .retrieve)
        llm:            the chat model (has .invoke)
        top_k:          how many chunks to retrieve
        min_score:      drop chunks less similar than this (0 to 1)
        return_context: if True, also return the full text sent to the LLM

    Returns:
        dict with: question, answer, sources, confidence, confidence_label
        (plus context, if return_context=True)
    """

    # ---- 1. Retrieve the closest chunks (each has content, metadata, score) ----
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    # ---- 2. Nothing relevant found: skip the LLM and say so ----
    if not results:
        output = {
            "question": query,
            "answer": "I don't know. No relevant information was found in the documents.",
            "sources": [],
            "confidence": 0.0,
            "confidence_label": "None",
        }
        if return_context:
            output["context"] = ""
        return output

    # ---- 3. Join the chunk texts into one context string ----
    context = "\n\n".join(doc["content"] for doc in results)

    # ---- 4. Collect unique source files, keeping their order ----
    sources = []
    for doc in results:
        source = doc["metadata"].get("source", "unknown")
        if source not in sources:
            sources.append(source)

    # ---- 5. Confidence: the best chunk's similarity score, as a percentage ----
    # Cosine similarity runs from 0 to 1, so the top score works as a simple confidence.
    best_score = max(doc["score"] for doc in results)
    confidence = round(max(0.0, min(best_score, 1.0)) * 100, 1)

    # Turn the number into a label that is easy to read
    if confidence >= 60:
        label = "High"
    elif confidence >= 40:
        label = "Medium"
    else:
        label = "Low"

    # ---- 6. Build the prompt: instructions + context + question ----
    prompt = f"""Answer using only the text below. If the answer is not there, say "I don't know".

Text:
{context}

Question:
{query}
"""

    # ---- 7. Ask the LLM and keep only the text of its reply ----
    answer = llm.invoke(prompt).content

    # ---- 8. Pack everything into one result ----
    output = {
        "question": query,
        "answer": answer,
        "sources": sources,
        "confidence": confidence,
        "confidence_label": label,
    }

    # Only include the full context when asked for it
    if return_context:
        output["context"] = context

    return output


# ------------------------------------------------------------
# Ask the user for a question (nothing is hardcoded)
# ------------------------------------------------------------
query = input("Ask a question: ").strip()

if not query:
    print("Please type a question.")
else:
    # Ask whether to show the full context (optional)
    show_context = input("Show full context? (y/n): ").strip().lower() == "y"

    result = rag_advanced(query, rag_retriever, llm, top_k=3, return_context=show_context)

    print("\nQuestion  :", result["question"])
    print("Answer    :", result["answer"])
    print("Sources   :", result["sources"])
    print(f"Confidence: {result['confidence']}% ({result['confidence_label']})")

    # Show the full context only if it was requested
    if "context" in result:
        print("\n--- Context sent to the LLM ---")
        print(result["context"])

In [26]:


print(rag_simple("What are the architecture decisions?", rag_retriever, llm, top_k=3))


Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.18it/s]

Generated embeddings with shape: (1, 384)


I don't know.
